1. Import the necessary packages

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk

2. loading the data set 
These are the order of the df.iloc numerical column index 
# ID, age, ho_incm, edu, BD1_11, BD2_31, BS3_1, BS3_2, BM7, BM14, O_TC

# starting from '300' is the additional smoking features used just for testing the NaN count to replace 'BS3_1' and 'BS3_2' 


In [ ]:
df = pd.read_csv('KNHANES9.csv', low_memory=False)

df_subset = (df.iloc[:, [1, 9, 12, 15, 232, 234, 520, 535, 496, 300]])
# columns:        ID  age  ho_incm  edu  BD1  BD1_11  BM7  BM14  O_TC  BS1_1

age_threshold = 50
df_subset = df_subset[df_subset['age'] >= age_threshold]
print(df_subset.head(1))

3. 
Counting the values of NaN etc.
a. it shows which values it will drop, considers which are NaN 
^ the value that its going to drop is ['-1.80E+308'] <-- classified as NaN  in the data set 
b. 'errors = 'coerce'' means that its going to take those ['-1.80E+308'] and 
    swap them to 'NaN' 
c. removes all rows where O_TC is NaN, keeps rows with valid tooth count
subset=['O_TC'] means that it only checks that column --> drops it / deletes
d.  creates a new binary label column for O_TC; 
    'astype(int) <-- computer views 1 = True; 0 = False 
    'O_TC_' <-- creates a new column 
e. 'label_count' = counts how many people are in each group 0 and 1 


In [36]:
print(df_subset[pd.to_numeric(df_subset['O_TC'], errors='coerce').isna()]['O_TC'].unique())

df_subset['O_TC'] = pd.to_numeric(df_subset['O_TC'], errors='coerce')
before_drop = len(df_subset)
df_subset = df_subset.dropna(subset=['O_TC'])
print(f"Rows droppped (NaN O_TC): {before_drop - len(df_subset)}")

df_subset['O_TC_'] = (df_subset['O_TC'] < 20).astype(int)

label_counts    = df_subset['O_TC_'].value_counts()
label_0         = label_counts[0]
label_1         = label_counts[1]
imbalance_ratio = label_0 / label_1

print(f"Age filter: >= {age_threshold}")
print(f"Label 0 (≥20 teeth): {label_0}")
print(f"Label 1 (<20 teeth): {label_1}")
print(df_subset['O_TC'].value_counts(dropna=False).head(20))
print(f"Imbalance ratio: {imbalance_ratio:.2f}")

print(f"NaN count: {df_subset['O_TC'].isna().sum()}")

['-1.80E+308']
Rows droppped (NaN O_TC): 410
Age filter: >= 50
Label 0 (≥20 teeth): 2441
Label 1 (<20 teeth): 894
O_TC
28.0    556
27.0    350
26.0    283
25.0    188
0.0     179
24.0    179
23.0    166
32.0    132
29.0    128
21.0    114
22.0    112
30.0     95
20.0     89
19.0     67
16.0     59
17.0     53
18.0     49
31.0     49
15.0     41
12.0     39
Name: count, dtype: int64
Imbalance ratio: 2.73
NaN count: 0


4. Missing Value Audit 

In [ ]:
invalid_codes = [8, 9, 88, 99, 888, 999, 8888, 9999]
feature_cols  = ['age', 'ho_incm', 'edu', 'BD1', 'BD1_11', 'BS1_1', 'BM7', 'BM14']

df_subset[feature_cols] = df_subset[feature_cols].replace(invalid_codes, np.nan)
df_subset[feature_cols] = df_subset[feature_cols].apply(pd.to_numeric, errors='coerce')
df_subset[feature_cols] = df_subset[feature_cols].where(df_subset[feature_cols].abs() < 1e10, np.nan)

print("NaN counts per column after cleaning:")
total = len(df_subset)
for col in feature_cols:
    count = df_subset[col].isna().sum()
    pct   = (count / total) * 100
    print(f"{col}: {count} | ({pct:.1f}%)")

# !! HIGH NaN COUNT !!

BS3_1 and BS3_2 has an extraordinarily high number of NaN values. This could cause a lot of issues in data handling and the overall accuracy. Moreover, imputing would not work as well. Therefore we need to drop these features and replace them with a similar feature (related to smoking). Below are features related to smoking and their NaN count: 

In [40]:
feature_cols2 = ['BS1_1', 'BS2_1', 'BS2_2', 'BS3_3', 'BS6_2', 'BS6_2_1', 'BS6_2_2', 'BS6_3', 'BS6_4', 'BS6_4_1', 'BS6_4_2']

df_subset[feature_cols2] = df_subset[feature_cols2].replace(invalid_codes, np.nan)
df_subset[feature_cols2] = df_subset[feature_cols2].apply(pd.to_numeric, errors='coerce')
df_subset[feature_cols2] = df_subset[feature_cols2].where(df_subset[feature_cols2].abs() < 1e10, np.nan)

print("NaN counts per column after cleaning (SMOKING FOCUSED):")
total = len(df_subset)
for col in feature_cols2:
    count = df_subset[col].isna().sum()
    pct = (count/total) * 100 
    print(f"{col}: {count} | ({pct:.1f}%)")

NaN counts per column after cleaning (SMOKING FOCUSED):
BS1_1: 134 | (4.0%)
BS2_1: 2144 | (64.3%)
BS2_2: 2194 | (65.8%)
BS3_3: 3295 | (98.8%)
BS6_2: 2530 | (75.9%)
BS6_2_1: 2539 | (76.1%)
BS6_2_2: 2530 | (75.9%)
BS6_3: 2533 | (76.0%)
BS6_4: 2528 | (75.8%)
BS6_4_1: 2544 | (76.3%)
BS6_4_2: 2533 | (76.0%)


# !! Smoking NaN Count !! 

However, even the other features related to smoking has a high count of NaN as well that can't be used in training nor imputing. All except for BS1_1. 

BS1_1 is a feature that asks "Lifetime conventional cigarette smoking status" 
    1. Less than 5 packs (100 cigarettes)
    2. 5 packs (100 cigarettes) or more
    3. Never smoked
    8. Not applicable (children, adolescents)
    9. Don't know, no response

The NaN count is 134 or 4.0% of the total answers. This is the most suitable therefore this should replace the 'BS3_1' and 'BS3_2' overall. 


5. Standardization 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = df_subset[['age', 'ho_incm', 'edu', 'BD1', 'BD1_11', 'BS1_1', 'BM7', 'BM14']]
y = df_subset['O_TC_']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

In [29]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

# For Logistic Regression and SVM — normalize all features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# For RF, XGBoost, LightGBM — NO standardization needed
X_train_tree = X_train.copy()
X_test_tree  = X_test.copy()

6. Class Imbalance -- using SMOTE 

In [30]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(k_neighbors=5, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_resampled).value_counts().to_dict())

Before SMOTE: {0: 1953, 1: 715}
After SMOTE:  {0: 1953, 1: 1953}
